# Día 6 - Optimización, Kaggle/OpenML y diseño del proyecto

**Fecha:** 20-10-2026

Hoy organizamos el proceso de mejorar un modelo sin “perseguir” Test.

## Objetivos

- distinguir parámetros e hiperparámetros;
- hacer búsqueda manual y GridSearchCV;
- visualizar cómo se mueve una métrica;
- descargar datasets desde Kaggle/OpenML;
- aplicar un checklist de calidad antes de modelar;
- definir el proyecto final.

## 0. Librerías y carpeta de datos

No hay instalación automática. Si una librería opcional falta, el notebook mostrará el comando que puedes ejecutar manualmente en una celda nueva. Todos los imports principales quedan juntos para que sea claro qué herramientas usa la jornada.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.datasets import fetch_openml

# KaggleHub es opcional. No se instala automáticamente.
try:
    import kagglehub
except ImportError:
    kagglehub = None

In [ ]:
DATA_DIR = Path('../datos')
if not DATA_DIR.exists():
    DATA_DIR = Path('datos')
print('Datos:', DATA_DIR.resolve())

A partir de aquí, cada bloque de código irá acompañado de una explicación de **qué entra, qué sale y qué debemos mirar**. Ejecutar una celda sin interpretar el resultado no cuenta como comprender el modelo.

# 1. Parámetro vs hiperparámetro

| Tipo | Ejemplo | Quién lo determina |
|---|---|---|
| Parámetro | coeficiente de regresión | entrenamiento |
| Parámetro | reglas de un árbol | entrenamiento |
| Hiperparámetro | `max_depth` | analista / búsqueda |
| Hiperparámetro | `n_estimators` | analista / búsqueda |
| Decisión de modelado | qué variables usar | analista |

Un hiperparámetro debe seleccionarse usando Validation o Cross-Validation, no mirando Test repetidamente.

Un **parámetro** lo aprende el modelo desde Train: coeficientes, cortes de un árbol, centroides, etc.

Un **hiperparámetro** define cómo se permite aprender: `max_depth`, `alpha`, `k`, `C`, `gamma`, `n_estimators`...

La búsqueda de hiperparámetros **forma parte del entrenamiento/model selection**, por eso se decide usando Validation o CV, no mirando Test repetidamente.

## 1.1 Mapa rápido de hiperparámetros

| Modelo | Hiperparámetros típicos |
|---|---|
| Linear | `fit_intercept` |
| Ridge/Lasso | `alpha` |
| Árbol | `max_depth`, `min_samples_leaf`, `ccp_alpha` |
| Random Forest | `n_estimators`, `max_depth`, `max_features` |
| KNN | `n_neighbors`, `weights`, `metric` |
| SVM | `C`, `kernel`, `gamma` |
| K-Means | `n_clusters`, `n_init` |

El hiperparámetro cambia el **comportamiento del algoritmo**, no es algo que el modelo aprende como un coeficiente.

# 2. Búsqueda manual: primero entender, después automatizar

Antes de `GridSearchCV`, conviene mover un parámetro a mano para ver su efecto.

Antes de automatizar, mover un hiperparámetro a mano permite entender su efecto. Queremos ver simultáneamente:

- error en Train;
- error en Validation;
- brecha entre ambos;
- costo computacional.

La curva de Validation suele ser más informativa para elegir complejidad que la mejora monotónica de Train.

In [ ]:
df=pd.read_csv(DATA_DIR/'actividad_economica_demo.csv').drop_duplicates()
for c in ['inflacion_pct','indice_confianza']:
    df[c]=df[c].fillna(df[c].median())

features=['inflacion_pct','desempleo_pct','exportaciones_mill','inversion_mill','precio_petroleo_usd','indice_confianza']
target='crecimiento_actividad_pct'

train_val,test=train_test_split(df,test_size=.15,random_state=42)
train,val=train_test_split(train_val,test_size=.1765,random_state=42)

In [ ]:
filas=[]

for profundidad in [2,3,4,5,7,10,None]:
    m=RandomForestRegressor(
        n_estimators=200,
        max_depth=profundidad,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )
    m.fit(train[features],train[target])
    pred_tr=m.predict(train[features])
    pred_val=m.predict(val[features])

    filas.append({
        'max_depth':str(profundidad),
        'MAE_train':mean_absolute_error(train[target],pred_tr),
        'MAE_val':mean_absolute_error(val[target],pred_val)
    })

busqueda_manual=pd.DataFrame(filas)
busqueda_manual.round(3)

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(busqueda_manual['max_depth'],busqueda_manual['MAE_train'],marker='o',label='Train')
plt.plot(busqueda_manual['max_depth'],busqueda_manual['MAE_val'],marker='o',label='Validation')
plt.ylabel('MAE')
plt.xlabel('max_depth')
plt.title('Complejidad vs error')
plt.legend()
plt.show()

### Cómo interpretar la curva

- si Train mejora y Validation también → todavía ganamos capacidad útil;
- si Train mejora pero Validation empeora → comenzamos a sobreajustar;
- si ambos son malos → modelo demasiado limitado o features insuficientes.

El punto óptimo no tiene por qué ser exactamente el mínimo de una sola ejecución; conviene usar CV y considerar estabilidad.

## 2.1 ¿Qué buscamos al mover complejidad?

No buscamos el menor error de Train. Buscamos una zona donde:

- Train sea razonable;
- Validation sea bajo;
- la brecha Train/Validation no sea extrema;
- el resultado sea estable.

In [ ]:
# Añadimos la brecha como herramienta de diagnóstico.
busqueda_manual['brecha_MAE']=busqueda_manual['MAE_val']-busqueda_manual['MAE_train']
busqueda_manual.round(3)

## 2.2 Dos hiperparámetros interactúan

`max_depth=10` puede comportarse diferente con `min_samples_leaf=1` que con `min_samples_leaf=20`. Por eso a veces necesitamos buscar combinaciones.

Los hiperparámetros no actúan aislados. Por ejemplo, un bosque con árboles profundos puede necesitar `min_samples_leaf` mayor; en boosting, `learning_rate` y `n_estimators` forman un compromiso. Por eso las búsquedas evalúan **combinaciones**, no parámetros independientes.

In [ ]:
filas=[]
for depth in [3,6,10,None]:
    for leaf in [1,5,15]:
        m=RandomForestRegressor(
            n_estimators=150,max_depth=depth,min_samples_leaf=leaf,
            random_state=42,n_jobs=-1
        )
        m.fit(train[features],train[target])
        filas.append({
            'depth':str(depth),'leaf':leaf,
            'MAE_val':mean_absolute_error(val[target],m.predict(val[features]))
        })

pd.DataFrame(filas).pivot(index='depth',columns='leaf',values='MAE_val').round(3)

# 3. GridSearchCV

GridSearchCV prueba combinaciones de hiperparámetros y evalúa cada combinación mediante cross-validation.

Ventaja: proceso sistemático.  
Riesgo: una grilla enorme consume tiempo y puede terminar sobreoptimizando el procedimiento de validación.

`GridSearchCV` prueba todas las combinaciones de una cuadrícula y evalúa cada una mediante cross-validation.

Flujo conceptual:

```text
Train
 ├─ fold 1 validation interna
 ├─ fold 2 validation interna
 ├─ ...
 └─ combinación de hiperparámetros con score medio
```

Nuestro Test externo sigue guardado. GridSearchCV **no debe ver Test**.

In [ ]:
param_grid={
    'n_estimators':[100,250],
    'max_depth':[3,5,8,None],
    'min_samples_leaf':[2,5,10]
}

base=RandomForestRegressor(random_state=42,n_jobs=-1)

grid=GridSearchCV(
    base,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    return_train_score=True
)

grid.fit(train[features],train[target])
print('Mejores hiperparámetros:',grid.best_params_)
print('MAE CV aproximado:',round(-grid.best_score_,3))

In [ ]:
# Evaluamos la configuración seleccionada en nuestro Validation externo.
best=grid.best_estimator_
pred_val=best.predict(val[features])

print('Validation MAE :',round(mean_absolute_error(val[target],pred_val),3))
print('Validation RMSE:',round(np.sqrt(mean_squared_error(val[target],pred_val)),3))
print('Validation R²  :',round(r2_score(val[target],pred_val),3))

## 3.1 Leer los resultados completos de GridSearchCV

`best_params_` es cómodo, pero conviene mirar más de una combinación: diferencias minúsculas pueden no justificar un modelo más complejo.

No mires únicamente `best_params_`. `cv_results_` muestra media y dispersión por combinación. Dos configuraciones con medias casi iguales pueden diferir en estabilidad o complejidad; normalmente preferimos la más simple si el rendimiento es equivalente.

In [ ]:
cv_res=pd.DataFrame(grid.cv_results_)
cols=['param_n_estimators','param_max_depth','param_min_samples_leaf','mean_train_score','mean_test_score','std_test_score','rank_test_score']
cv_res[cols].sort_values('rank_test_score').head(10)

### ¿Por qué los scores son negativos?

Scikit-learn usa la convención “más grande es mejor” en sus funciones de scoring. Para errores como MAE, almacena `neg_mean_absolute_error`. Multiplicar por -1 devuelve el MAE intuitivo.

Scikit-learn sigue la convención “mayor score = mejor”. Para errores que naturalmente queremos minimizar, como MAE, expone `neg_mean_absolute_error`. Por eso aparecen valores negativos durante la búsqueda; para interpretarlos como error, multiplicamos por -1.

In [ ]:
cv_res['MAE_cv']=-cv_res['mean_test_score']
cv_res[['MAE_cv','std_test_score','rank_test_score']].sort_values('rank_test_score').head().round(3)

# 4. Kaggle: repositorio de datasets, no algoritmo

Podemos trabajar de dos formas:

### Manual
1. buscar dataset;
2. descargar ZIP;
3. descomprimir;
4. leer CSV con Pandas.

### Programática
`kagglehub` puede descargar datasets públicos desde código.

**Nunca subas datos institucionales o confidenciales a Kaggle.**

Kaggle es un repositorio/plataforma, no una librería de ML. El reto pedagógico no es “hacer download”, sino aprender a preguntar antes de modelar:

- ¿quién creó el dataset?
- ¿qué representa una fila?
- ¿qué significa el target?
- ¿hay licencia/fecha/documentación?
- ¿existe leakage?
- ¿el dataset refleja el problema real o solo un benchmark?

El ZIP incluye un respaldo local didáctico para que la clase no dependa de Internet.

## 3.2 ¿Cuándo usar RandomizedSearchCV?

Una cuadrícula grande explota combinatoriamente. `RandomizedSearchCV` prueba una cantidad limitada de combinaciones muestreadas. Es útil cuando:

- hay muchos hiperparámetros;
- cada entrenamiento es costoso;
- no conocemos un rango estrecho razonable.

Después podemos hacer una búsqueda más fina alrededor de las zonas prometedoras.

In [ ]:
# Ejemplo compacto de búsqueda aleatoria sobre Random Forest.
param_dist = {
    'n_estimators': [100, 200, 400, 700],
    'max_depth': [None, 3, 5, 8, 12],
    'min_samples_leaf': [1, 2, 4, 8, 15],
    'max_features': ['sqrt', 0.5, 1.0]
}

busqueda_random = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=12,
    scoring='neg_mean_absolute_error',
    cv=4,
    random_state=42,
    n_jobs=-1
)

busqueda_random.fit(train[features], train[target])
print('Mejores parámetros:', busqueda_random.best_params_)
print('MAE CV estimado:', round(-busqueda_random.best_score_, 3))

El resultado es una **estimación interna sobre Train mediante CV**. Todavía no es el resultado final en Test.

In [ ]:
# Descarga programática OPCIONAL. Si kagglehub no está instalado, seguimos con el CSV local.
if kagglehub is None:
    print('kagglehub no está instalado. Si quieres usarlo: %pip install kagglehub')
    print('Para la clase usaremos datos/telco_churn_fallback.csv como respaldo.')
else:
    try:
        ruta = kagglehub.dataset_download('blastchar/telco-customer-churn')
        print('Dataset descargado en:', ruta)
    except Exception as e:
        print('No se pudo descargar Kaggle en este momento:', type(e).__name__)
        print('Usaremos el respaldo local.')

In [ ]:
# Fallback local garantizado para la práctica.
telco=pd.read_csv(DATA_DIR/'telco_churn_fallback.csv')

print(telco.shape)
telco.head()

## 4.1 Antes de modelar un dataset de Kaggle

No ejecutes un modelo inmediatamente. Responde:

1. ¿qué representa una fila?
2. ¿quién creó el dataset?
3. ¿cuál es la licencia/origen?
4. ¿cuál es el target?
5. ¿existe leakage?
6. ¿qué clases hay y están balanceadas?
7. ¿qué tipos de columnas existen?
8. ¿hay nulos/duplicados/outliers?

Este paso no debe reducirse a `df.head()`. Como mínimo documenta: unidad de observación, target, tipos, nulos, duplicados, balance del target, variables disponibles al momento de predecir y posibles identificadores que no deberían entrar al modelo.

In [ ]:
print(telco.info())
print()
print('Target:')
print(telco['Churn'].value_counts(normalize=True).round(3))
print()
print('Nulos:')
print(telco.isna().sum().sort_values(ascending=False).head())

## 4.2 Kaggle descargado manualmente

Si no quieres usar API ni credenciales, el flujo más universal sigue siendo:

```python
ruta = 'datos/archivo_kaggle.csv'
df = pd.read_csv(ruta)
```

Lo importante es registrar fuente, fecha de descarga, licencia y significado de columnas. La sofisticación de la descarga no mejora el modelo.

In [ ]:
# Ejemplo local del mismo flujo.
print('Filas:',len(telco))
print('Columnas:',telco.columns.tolist())
print('Duplicados:',telco.duplicated().sum())

# 5. OpenML

OpenML permite descargar datasets directamente desde Scikit-learn mediante `fetch_openml`.

Ventaja didáctica: obtenemos X e y sin buscar manualmente el archivo.

OpenML facilita descargar datasets y metadatos directamente desde código. Es cómodo para experimentos reproducibles, pero sigue aplicando la misma regla: **la facilidad de acceso no sustituye entender el origen y significado de las columnas**.

In [ ]:
# Pon True si quieres hacer la descarga EN VIVO durante la clase.
# En False, el notebook no depende de Internet.
DESCARGAR_OPENML = False

if DESCARGAR_OPENML:
    try:
        credit=fetch_openml(name='credit-g',version=1,as_frame=True)
        X_credit=credit.data
        y_credit=credit.target
        print(X_credit.shape)
        display(X_credit.head())
        print(y_credit.value_counts())
    except Exception as e:
        print('OpenML no disponible:',e)
else:
    print('Descarga OpenML desactivada. Cambia DESCARGAR_OPENML=True para probarla en vivo.')

## 5.1 OpenML también necesita EDA

Aunque `fetch_openml` entregue `data` y `target` separados, seguimos necesitando revisar tipos, categorías, nulos y distribución de clases. “Viene preparado” no significa “está listo para cualquier modelo”.

## 5.2 Fuentes IMF, CEPAL, BCE e INEC

No todas las fuentes tienen la misma interfaz. La estrategia práctica es:

1. descargar/API;
2. guardar copia local con fecha de corte;
3. documentar metadata;
4. transformar a una tabla donde cada fila tenga significado claro;
5. recién entonces pensar en ML.

En fuentes SDMX como IMF, la estructura de series y dimensiones es parte del problema de datos; no conviene ocultarla detrás de un notebook “mágico”.

Para fuentes estadísticas oficiales, además de cargar datos debemos documentar:

- periodicidad (mensual, trimestral, anual);
- unidades/precios corrientes o constantes;
- revisiones metodológicas;
- cobertura geográfica;
- valores preliminares/definitivos;
- fecha de disponibilidad real para evitar leakage temporal.

En macroeconomía, estas decisiones pueden importar tanto como el algoritmo.

# 6. Reto: escoger un dataset apropiado

Puedes utilizar:

- Banco Mundial;
- IMF;
- CEPALSTAT;
- INEC;
- BCE;
- Kaggle;
- UCI;
- OpenML.

Pero debes justificarlo con estas preguntas:

1. ¿de dónde proviene?
2. ¿qué representa una fila?
3. ¿qué target existe o qué estructura quiero descubrir?
4. ¿cuántas observaciones tengo?
5. ¿qué tipos de variables hay?
6. ¿qué datos faltan?
7. ¿es regresión, clasificación o clustering?
8. ¿qué baseline usaré?
9. ¿qué métrica representa mejor el costo del error?
10. ¿cómo separaré Train/Validation/Test?

El objetivo del reto no es buscar el CSV más grande. Es justificar por qué el dataset permite responder una pregunta concreta de regresión, clasificación o clustering y qué limitaciones tendría una conclusión.

## 6.1 Ficha de decisión del dataset

Antes de aprobar un dataset para el proyecto, completa esta ficha:

| Pregunta | Respuesta |
|---|---|
| Unidad de observación | |
| Target / objetivo | |
| Momento de predicción | |
| Variables disponibles en ese momento | |
| Tamaño | |
| Nulos / duplicados | |
| Posible leakage | |
| Baseline | |
| Métrica principal | |
| Split | |

Esta tabla evita empezar por “qué algoritmo quiero usar” antes de definir el problema.

# 7. Taller U4 - optimización

Entrega breve:

- modelo base;
- hiperparámetros probados;
- tabla Train/CV/Validation;
- mejor configuración justificada;
- explicación de por qué Test todavía no se usa;
- una reflexión sobre overfitting de hiperparámetros.

## Preguntas para cerrar

1. ¿Qué diferencia existe entre parámetro e hiperparámetro?
2. ¿Por qué no se debe optimizar mirando Test?
3. ¿Qué patrón Train/Validation sugiere overfitting?
4. ¿Qué ventaja tiene RandomizedSearch frente a una cuadrícula enorme?
5. ¿Por qué `best_params_` no garantiza el mejor modelo futuro?
6. ¿Qué debes investigar de un dataset de Kaggle antes de modelarlo?
7. ¿Qué particularidades tienen los datos macroeconómicos respecto a fecha de publicación y revisiones?